# Full GLUE Evaluation - morphology_clean_fine_tuned Model

Comprehensive GLUE benchmark evaluation on all 6 tasks:
- MRPC (Microsoft Research Paraphrase Corpus)
- QQP (Quora Question Pairs)
- BoolQ (Boolean Questions)
- SST-2 (Stanford Sentiment Treebank)
- RTE (Recognizing Textual Entailment)
- CoLA (Corpus of Linguistic Acceptability)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install dependencies
!pip install -q sentencepiece datasets transformers torch

In [ ]:
# Upload your model to Google Drive first, then set the path here
# Or zip and upload the morphology_clean_fine_tuned directory

MODEL_PATH = "/content/drive/MyDrive/BabyLM/outputs/morphology_contrastive_finetuned/final_model3"  # Adjust this path

import os
if not os.path.exists(MODEL_PATH):
    print("❌ Model not found! Please upload morphology_clean_fine_tuned to Google Drive")
    print(f"   Looking for: {MODEL_PATH}")
else:
    print(f"✅ Model found: {MODEL_PATH}")
    print(f"   Files: {os.listdir(MODEL_PATH)}")

✅ Model found: /content/drive/MyDrive/BabyLM/outputs/morphology_contrastive_finetuned/final_model3
   Files: ['model.safetensors', 'generation_config.json', 'config.json', 'spm.model']


In [ ]:
import zipfile
import os

zip_file_path = '/content/drive/MyDrive/BabyLM/morphology_contrastive_finetuned.zip'# Your path
unzip_dir_name = 'morphology_contrastive_finetuned' # Your model
target_dir_name = 'model_to_eval'

# Unzip the file
print(f"Unzipping {zip_file_path}...")
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall('/content/drive/MyDrive/BabyLM/')
print(f"Unzipped to /content/drive/MyDrive/BabyLM/{unzip_dir_name}")

# Rename the unzipped directory
old_path = os.path.join('/content/drive/MyDrive/BabyLM/', unzip_dir_name)
new_path = os.path.join('/content/drive/MyDrive/BabyLM/', target_dir_name)

if os.path.exists(old_path):
    os.rename(old_path, new_path)
    print(f"Renamed '{unzip_dir_name}' to '{target_dir_name}'")
else:
    print(f"Directory '{old_path}' not found after unzipping.")

# Update MODEL_PATH
MODEL_PATH = new_path
print(f"Updated MODEL_PATH to: {MODEL_PATH}")

Unzipping /content/drive/MyDrive/BabyLM/morphology_contrastive_finetuned2.zip...
✅ Unzipped to /content/drive/MyDrive/BabyLM/morphology_contrastive_finetuned2
❌ Directory '/content/drive/MyDrive/BabyLM/morphology_contrastive_finetuned2' not found after unzipping.
Updated MODEL_PATH to: /content/drive/MyDrive/BabyLM/model_to_eval


In [ ]:
# Complete evaluation script
import json
import datasets
import torch
import torch.nn as nn
import sentencepiece as spm
from transformers import GPT2LMHeadModel, GPT2Config
from functools import partial
from tqdm import tqdm

# GLUE tasks
GLUE_TASKS = ['mrpc', 'qqp', 'boolq', 'sst2', 'rte', 'cola']

TASK_INPUTS = {
    'mrpc': ['sentence1', 'sentence2'],
    'qqp': ['question1', 'question2'],
    'boolq': ['question', 'passage'],
    'sst2': ['sentence'],
    'rte': ['sentence1', 'sentence2'],
    'cola': ['sentence'],
}

TASK_NUM_LABELS = {
    'mrpc': 2, 'qqp': 2, 'boolq': 2,
    'sst2': 2, 'rte': 2, 'cola': 2,
}

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

class SentencePieceTokenizerWrapper:
    def __init__(self, model_path):
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)
        self.vocab_size = self.sp.vocab_size()
        self.pad_token_id = self.sp.pad_id()
        self.model_max_length = 512

    def encode(self, text, truncation=True, max_length=None):
        max_len = max_length or self.model_max_length
        ids = self.sp.encode(text, out_type=int)
        if truncation and len(ids) > max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids):
        return self.sp.decode(ids)

class GPT2ForSequenceClassification(nn.Module):
    def __init__(self, base_model_path, num_labels=2):
        super().__init__()
        self.transformer = GPT2LMHeadModel.from_pretrained(base_model_path).transformer
        self.config = self.transformer.config
        self.score = nn.Linear(self.config.n_embd, num_labels, bias=False)
        self.num_labels = num_labels

    def forward(self, input_ids, attention_mask=None, labels=None):
        transformer_outputs = self.transformer(input_ids, attention_mask=attention_mask)
        hidden_states = transformer_outputs[0]

        if attention_mask is not None:
            sequence_lengths = attention_mask.sum(dim=1) - 1
        else:
            sequence_lengths = torch.full((input_ids.shape[0],), input_ids.shape[1] - 1, device=input_ids.device)

        pooled_hidden_states = hidden_states[torch.arange(hidden_states.shape[0], device=hidden_states.device), sequence_lengths]
        logits = self.score(pooled_hidden_states)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits, labels)

        return {'loss': loss, 'logits': logits}

def tokenize_function(examples, tokenizer, task):
    batch = {"input_ids": [], "labels": []}
    input_fields = TASK_INPUTS[task]

    for i in range(len(examples['label'])):
        input_texts = [examples[field][i] for field in input_fields]
        input_text = " ".join(input_texts)
        input_ids = tokenizer.encode(input_text, truncation=True)
        batch["input_ids"].append(input_ids)
        batch["labels"].append(examples['label'][i])

    return batch

def padding_collate_fn(batch, max_len=512, pad_token_id=0):
    max_length = min(max_len, max([len(item['input_ids']) for item in batch]))
    batch_size = len(batch)
    padded_input_ids = torch.full((batch_size, max_length), pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((batch_size, max_length), dtype=torch.long)
    labels = torch.tensor([item['labels'] for item in batch], dtype=torch.long)

    for i, item in enumerate(batch):
        seq_len = min(len(item['input_ids']), max_length)
        padded_input_ids[i, :seq_len] = torch.tensor(item['input_ids'][:seq_len])
        attention_mask[i, :seq_len] = 1

    return {'input_ids': padded_input_ids, 'attention_mask': attention_mask, 'labels': labels}

def evaluate(model, dataloader, task):
    correct, total = 0, 0
    tp, fp, fn = 0, 0, 0

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids, attention_mask=attention_mask)
            predictions = outputs['logits'].argmax(dim=-1)

            correct += (predictions == labels).sum().item()
            total += labels.shape[0]

            tp += ((predictions == 1) & (labels == 1)).sum().item()
            fp += ((predictions == 1) & (labels == 0)).sum().item()
            fn += ((predictions == 0) & (labels == 1)).sum().item()

    if task == 'mrpc':
        f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0.0
        return f1
    else:
        accuracy = correct / total if total > 0 else 0.0
        return accuracy

def train_and_evaluate(model, tokenizer, task, batch_size=16, lr=2e-5, max_epochs=5, patience=2, train_size=100000, val_size=10000):
    print(f"\n{'='*70}")
    print(f"Evaluating on {task.upper()}")
    print(f"{'='*70}\n")

    # Load dataset
    if task in ['boolq']:
        dataset = datasets.load_dataset('super_glue', task)
    else:
        task_name = 'sst2' if task == 'sst2' else task
        dataset = datasets.load_dataset('glue', task_name)

    # Limit dataset size
    train_s = min(train_size, len(dataset['train']))
    val_s = min(val_size, len(dataset['validation']))

    dataset['train'] = dataset['train'].select(range(train_s))
    dataset['validation'] = dataset['validation'].select(range(val_s))

    print(f"Train size: {train_s}, Validation size: {val_s}")

    # Tokenize
    tokenize_fn = partial(tokenize_function, tokenizer=tokenizer, task=task)
    dataset = dataset.map(tokenize_fn, batched=True, num_proc=2, remove_columns=dataset['train'].column_names)

    # Create dataloaders
    collate_fn = partial(padding_collate_fn, max_len=512, pad_token_id=tokenizer.pad_token_id)
    train_dataloader = torch.utils.data.DataLoader(dataset['train'], batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    val_dataloader = torch.utils.data.DataLoader(dataset['validation'], batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

    # Setup optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)

    # Training loop
    best_metric = 0.0
    patience_counter = 0

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0

        for batch in tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{max_epochs}"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs['loss']

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        avg_train_loss = train_loss / len(train_dataloader)

        # Evaluate
        model.eval()
        metric = evaluate(model, val_dataloader, task)

        metric_name = "F1" if task == 'mrpc' else "Accuracy"
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val {metric_name}: {metric:.4f}")

        # Early stopping
        if metric > best_metric:
            best_metric = metric
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nBest {metric_name}: {best_metric:.4f}")
    return best_metric

print("All functions loaded successfully!")

Using device: cuda
✅ All functions loaded successfully!


In [ ]:
# Load model and tokenizer
print(f"Loading model from {MODEL_PATH}")

# Find tokenizer
tokenizer_path = f"{MODEL_PATH}/spm.model"
print(f"Tokenizer: {tokenizer_path}")

tokenizer = SentencePieceTokenizerWrapper(tokenizer_path)
print(f"Loaded tokenizer: vocab_size={tokenizer.vocab_size}")

Loading model from /content/drive/MyDrive/BabyLM/outputs/morphology_contrastive_finetuned/final_model3
Tokenizer: /content/drive/MyDrive/BabyLM/outputs/morphology_contrastive_finetuned/final_model3/spm.model
✅ Loaded tokenizer: vocab_size=10000


In [ ]:
# Run evaluation on ALL tasks
results = {}

for task in GLUE_TASKS:
    print(f"\n{'='*70}")
    print(f"STARTING TASK: {task.upper()}")
    print(f"{'='*70}")

    # Create model for this task
    model = GPT2ForSequenceClassification(MODEL_PATH, num_labels=TASK_NUM_LABELS[task]).to(DEVICE)
    print(f"Model loaded with {sum(p.numel() for p in model.parameters()):,} parameters")

    # Train and evaluate with full data
    metric = train_and_evaluate(
        model,
        tokenizer,
        task,
        batch_size=16,  # Can use larger batch on GPU
        lr=2e-5,
        max_epochs=5,
        patience=2,
        train_size=100000,  # FULL training data
        val_size=10000      # FULL validation data
    )

    results[task] = float(metric)

    # Clean up
    del model
    torch.cuda.empty_cache()

    print(f"{task.upper()} completed: {metric:.4f}")

print(f"\n{'='*70}")
print("ALL TASKS COMPLETED!")
print(f"{'='*70}")


STARTING TASK: MRPC
Model loaded with 25,528,320 parameters

Evaluating on MRPC



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrpc/train-00000-of-00001.parquet:   0%|          | 0.00/649k [00:00<?, ?B/s]

mrpc/validation-00000-of-00001.parquet:   0%|          | 0.00/75.7k [00:00<?, ?B/s]

...

Epoch 2: Train Loss: 0.5756, Val Accuracy: 0.6913


Epoch 3/5: 100%|██████████| 535/535 [00:24<00:00, 21.98it/s]


Epoch 3: Train Loss: 0.5367, Val Accuracy: 0.6807


Epoch 4/5: 100%|██████████| 535/535 [00:24<00:00, 22.27it/s]


Epoch 4: Train Loss: 0.4687, Val Accuracy: 0.6558
Early stopping at epoch 4

Best Accuracy: 0.6913
✅ COLA completed: 0.6913

ALL TASKS COMPLETED!


In [ ]:
# Display final results
print("\n" + "="*70)
print("FINAL GLUE RESULTS - morphology_clean_fine_tuned")
print("="*70 + "\n")

for task, score in results.items():
    metric_name = "F1" if task == 'mrpc' else "Accuracy"
    print(f"{task.upper():.<20} {score:.4f} ({metric_name})")

# Calculate average
avg_score = sum(results.values()) / len(results)
print(f"\n{'='*70}")
print(f"AVERAGE SCORE: {avg_score:.4f}")
print(f"{'='*70}")

# Save results
output_file = "/content/drive/MyDrive/BabyLM/glue_fine_tuned_results.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\nResults saved to: {output_file}")


FINAL GLUE RESULTS - morphology_clean_fine_tuned

MRPC................ 0.8108 (F1)
QQP................. 0.7785 (Accuracy)
BOOLQ............... 0.6633 (Accuracy)
SST2................ 0.8326 (Accuracy)
RTE................. 0.5199 (Accuracy)
COLA................ 0.6913 (Accuracy)

AVERAGE SCORE: 0.7161

✅ Results saved to: /content/drive/MyDrive/BabyLM/glue_fine_tuned_results.json


## Download Results

Results are automatically saved to your Google Drive. You can also download them directly:

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/BabyLM/glue_fine_tuned_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>